In [ ]:
import sqlite3
import pandas as pd
import os

# Define file paths relative to the notebook location
DB_PATH = "../m5_forecasting.db"
DATA_RAW_DIR = "../data_raw/"
DATA_PROC_DIR = "../data_processed/"

print("Connecting to the SQLite database...")
# Establish connection to the local SQLite database file
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Enforce foreign key constraints within the SQLite engine session
cursor.execute("PRAGMA foreign_keys = ON;")

def ingest_csv_to_sql(csv_path, table_name, chunk_driven=False):
    """
    Reads a CSV file and appends the records straight into the target SQL table.
    Uses chunk-driven streaming for massive tables to save system memory.
    """
    if not os.path.exists(csv_path):
        print(f"Error: Could not locate the file at {csv_path}")
        return

    print(f"Starting ingestion for table: '{table_name}' from {csv_path}...")
    
    if chunk_driven:
        # Processing in blocks of 500,000 rows to optimize RAM consumption
        chunk_size = 500000
        for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunk_size)):
            chunk.to_sql(table_name, conn, if_exists="append", index=False)
            print(f"   --> Processed batch {i + 1} ({chunk_size * (i + 1)} rows completed)")
    else:
        # Standard load for lighter dimension tables
        df = pd.read_csv(csv_path)
        df.to_sql(table_name, conn, if_exists="append", index=False)
        print(f"Successfully loaded '{table_name}' table.")

# --- INGESTION EXECUTION PIPELINE ---

# 1. Ingest dim_calendar
ingest_csv_to_sql(os.path.join(DATA_RAW_DIR, "calendar.csv"), "dim_calendar")

# 2. Ingest dim_sell_prices
ingest_csv_to_sql(os.path.join(DATA_RAW_DIR, "sell_prices.csv"), "dim_sell_prices")

# 3. Ingest fact_sales (Activating chunk-driven processing for 58 million rows)
ingest_csv_to_sql(
    os.path.join(DATA_PROC_DIR, "sales_unpivoted.csv"), 
    "fact_sales", 
    chunk_driven=True
)

# Commit changes and shut down the database session cleanly
conn.commit()
conn.close()
print("\nDatabase Ingestion Phase is Complete!")